T10 : modèles GARCH à h = 1, 5 et 22 (S&P 500 et TSX)

Objectif : prévoir la variance à 1, 5 et 22 jours avec GARCH(1,1), GJR-GARCH et EGARCH en loi de Student, sur le S&P 500 et le TSX, en walk-forward (ré-estimation tous les 22 jours, avec les seules données disponibles à la date de prévision).
Prévision à h jours : moyenne des variances prévues sur les h jours suivants, multipliée par le facteur c = rv.mean() / (ret²).mean(), recalculé à chaque ré-estimation. Pour EGARCH, il n'y a pas de formule exacte au-delà d'un jour : on simule 20 000 trajectoires (graine fixée).
Test : du 2015-01-02 au 2026-09-10 (SPX) ou 2026-09-09 (TSX). Estimation initiale 2000–2014.

In [1]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
from arch import arch_model

warnings.filterwarnings("ignore")

here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p/"data"/"features_spx.csv").exists())
sys.path.append(str(root))

from src.utils import TEST_START, save_forecasts, evaluate, qlike, mse, dm_test, load_all_forecasts

DATA = {k: pd.read_csv(root/"data"/f"{k}.csv", parse_dates=["Date"]).set_index("Date")
        for k in ("spx", "tsx")}
FEAT = {k: pd.read_csv(root/"data"/f"features_{k}.csv", index_col=0, parse_dates=True)
        for k in ("spx", "tsx")}

for k in ("spx", "tsx"):
    print(k, "| données :", DATA[k].index.min().date(), "→", DATA[k].index.max().date(),
          "| dernière cible h=1 :", FEAT[k]["y_h1"].dropna().index[-1].date(),
          "| dernière cible h=22 :", FEAT[k]["y_h22"].dropna().index[-1].date())

spx | données : 2000-01-04 → 2026-10-12 | dernière cible h=1 : 2026-09-10 | dernière cible h=22 : 2026-09-10
tsx | données : 2000-01-05 → 2026-10-09 | dernière cible h=1 : 2026-09-09 | dernière cible h=22 : 2026-09-09


In [2]:
from arch.univariate import StudentsT

SPECS = {
    "GARCH":  dict(vol="GARCH",  p=1, q=1),
    "GJR":    dict(vol="GARCH",  p=1, o=1, q=1),
    "EGARCH": dict(vol="EGARCH", p=1, o=1, q=1),
}
HORIZONS = [1, 5, 22]

def garch_walk_forward(index, name, refit=22, simulations=20000):
    data, feat = DATA[index], FEAT[index]
    ret = data["ret"]
    mk  = lambda x: arch_model(x, mean="Constant", dist="t", rescale=False, **SPECS[name])

    # dates de test : celles où la cible la plus lointaine (h=22) est connue
    dates = feat.index[(feat.index >= TEST_START) & feat["y_h22"].notna()]
    out, pas_converge = [], 0

    for k in range(0, len(dates), refit):
        block = dates[k:k + refit]
        d0    = block[0]
        train = data.loc[:d0]                              # rien après d0 pour estimer

        res = mk(train["ret"]).fit(disp="off")
        pas_converge += (res.convergence_flag != 0)
        c = train["rv"].mean() / (train["ret"]**2).mean()  # c recalculé

        modele = mk(ret.loc[:block[-1]]).fix(res.params)   # paramètres fixes sur 22 jours
        i0 = ret.index.get_loc(d0)

        if name == "EGARCH":
            # pas de formule exacte au-delà d'un jour : on simule
            rng = StudentsT(seed=42 + k).simulate([res.params["nu"]])
            v = modele.forecast(horizon=22, start=i0, method="simulation",
                                simulations=simulations, rng=rng, reindex=False).variance
        else:
            v = modele.forecast(horizon=22, start=i0, method="analytic",
                                reindex=False).variance

        v = v.iloc[:len(block)]            # une ligne par jour du bloc, 22 colonnes (h=1..22)
        assert (v.index == block).all()

        for h in HORIZONS:
            # prévision à h = moyenne des variances prévues sur les h jours suivants
            out.append(pd.DataFrame({
                "date": block, "model": name, "horizon": h,
                "forecast": c * v.iloc[:, :h].mean(axis=1).values,
                "actual": feat.loc[block, f"y_h{h}"].values}))

    print(index, name, "| non convergés :", pas_converge, flush=True)
    return pd.concat(out, ignore_index=True)

RES = {(ix, nom): garch_walk_forward(ix, nom) for ix in ("spx", "tsx") for nom in SPECS}

for (ix, nom), df in RES.items():
    print(ix, nom, len(df), df["date"].min().date(), "→", df["date"].max().date(),
          "| forecast min :", round(df["forecast"].min(), 4))

spx GARCH | non convergés : 0
spx GJR | non convergés : 0
spx EGARCH | non convergés : 0
tsx GARCH | non convergés : 0
tsx GJR | non convergés : 0
tsx EGARCH | non convergés : 0
spx GARCH 8817 2015-01-02 → 2026-09-10 | forecast min : 0.0801
spx GJR 8817 2015-01-02 → 2026-09-10 | forecast min : 0.0892
spx EGARCH 8817 2015-01-02 → 2026-09-10 | forecast min : 0.034
tsx GARCH 8799 2015-01-02 → 2026-09-09 | forecast min : 0.0808
tsx GJR 8799 2015-01-02 → 2026-09-09 | forecast min : 0.0804
tsx EGARCH 8799 2015-01-02 → 2026-09-09 | forecast min : 0.0507


In [3]:
# 1) un fichier par modèle et par indice, au format commun, 3 horizons dedans
for (ix, nom), df in RES.items():
    save_forecasts(df, nom, ix)        # results/forecasts_<ix>_<nom>.csv

# 2) comparaison de tous les modèles sauvegardés
pd.set_option("display.width", 200)
for ix in ("spx", "tsx"):
    print("=====", ix.upper())
    display(evaluate(ix))

===== SPX


,horizon,model,QLIKE,MSE,DM vs HAR,p-value
0,1,EGARCH,0.4060,1.1546,0.5520,0.5810
1,1,GARCH,0.4210,1.8802,2.7611,0.0058
2,1,GJR,0.3939,1.8964,-0.1583,0.8742
3,1,HAR,0.3960,1.3104,NaN,NaN
4,1,LGBM,0.3421,1.1248,-6.4970,0.0000
5,1,LGBM_noVIX,0.3671,1.2863,-4.0766,0.0000
6,5,EGARCH,0.3135,0.8471,1.3965,0.1626
7,5,GARCH,0.3035,1.8160,2.2989,0.0215
8,5,GJR,0.2907,1.8757,0.8485,0.3961
9,5,HAR,0.2754,0.9952,NaN,NaN


===== TSX


,horizon,model,QLIKE,MSE,DM vs HAR,p-value
0,1,EGARCH,0.3336,1.0661,-0.8049,0.4209
1,1,GARCH,0.3529,1.7221,1.2136,0.2249
2,1,GJR,0.3377,1.7421,-0.4054,0.6852
3,1,HAR,0.3416,1.1488,NaN,NaN
4,1,LGBM,0.2968,0.9444,-8.0151,0.0000
5,1,LGBM_noVIX,0.3020,1.1134,-6.9295,0.0000
6,5,EGARCH,0.2145,0.8761,-1.8708,0.0614
7,5,GARCH,0.2279,1.8016,-0.9315,0.3516
8,5,GJR,0.2183,1.7253,-1.6796,0.0930
9,5,HAR,0.2384,0.9970,NaN,NaN


Résultats. À h = 1 et 5, LightGBM est nettement meilleur que le HAR sur les deux indices (p < 0,001). Aucun GARCH ne bat le HAR de façon significative ; sur le S&P 500, le GARCH(1,1) est même significativement moins bon à h = 1 et 5 (p = 0,006 et 0,022). À h = 22, aucun modèle n'est significativement meilleur que le HAR. Sur le TSX, EGARCH et GJR sont proches de la significativité à h = 5 (p = 0,06 et 0,09). Le MSE des GARCH à h = 22 est très élevé (2,4 à 2,8 contre 0,9 pour le HAR) : après un choc, leurs prévisions restent hautes très longtemps, car la persistance est proche de 1.

In [5]:
def wf_h1(index, name, dist, refit=22):
    """Walk-forward h=1 (sert seulement à comparer loi normale / Student)."""
    data, feat = DATA[index], FEAT[index]
    ret = data["ret"]
    mk  = lambda x: arch_model(x, mean="Constant", dist=dist, rescale=False, **SPECS[name])
    dates = feat.index[(feat.index >= TEST_START) & feat["y_h22"].notna()]
    out = []
    for k in range(0, len(dates), refit):
        block = dates[k:k + refit]; d0 = block[0]
        train = data.loc[:d0]
        res = mk(train["ret"]).fit(disp="off")
        c = train["rv"].mean() / (train["ret"]**2).mean()
        v = mk(ret.loc[:block[-1]]).fix(res.params).forecast(
                horizon=1, start=ret.index.get_loc(d0), reindex=False
            ).variance.iloc[:len(block), 0]
        out.append(pd.DataFrame({"forecast": v.values * c,
                                 "actual": feat.loc[block, "y_h1"].values}))
    return pd.concat(out)

lignes = []
for ix in ("spx", "tsx"):
    tr = DATA[ix].loc[:"2014-12-31"]                      # estimation initiale 2000–2014
    for nom in SPECS:
        for loi, lab in [("normal", "Normale"), ("t", "Student")]:
            r = arch_model(tr["ret"], mean="Constant", dist=loi,
                           rescale=False, **SPECS[nom]).fit(disp="off")
            p = r.params
            a, g, b = p.get("alpha[1]", np.nan), p.get("gamma[1]", np.nan), p["beta[1]"]
            pers = a + b if nom == "GARCH" else (a + g/2 + b if nom == "GJR" else b)
            fc = wf_h1(ix, nom, loi)
            lignes.append({"indice": ix, "modèle": nom, "loi": lab,
                           "alpha": a, "gamma": g,
                           "t_gamma": r.tvalues.get("gamma[1]", np.nan),
                           "beta": b, "persistance": pers, "nu": p.get("nu", np.nan),
                           "AIC": r.aic,
                           "QLIKE_h1": qlike(fc["actual"], fc["forecast"]),
                           "MSE_h1":   mse(fc["actual"], fc["forecast"])})

table3 = pd.DataFrame(lignes).round(4)
table3.to_csv(root/"results"/"table3_garch.csv", index=False)
display(table3)

,indice,modèle,loi,alpha,gamma,t_gamma,beta,persistance,nu,AIC,QLIKE_h1,MSE_h1
0,spx,GARCH,Normale,0.0929,NaN,NaN,0.8951,0.9880,NaN,10825.9726,0.4175,1.6246
1,spx,GARCH,Student,0.0912,NaN,NaN,0.9025,0.9937,7.6251,10727.9502,0.4210,1.8802
2,spx,GJR,Normale,0.0000,0.1625,7.5897,0.9026,0.9839,NaN,10656.5399,0.3882,1.7053
3,spx,GJR,Student,0.0000,0.1635,7.5686,0.9054,0.9872,9.2708,10589.0870,0.3939,1.8964
4,spx,EGARCH,Normale,0.1107,-0.1427,-9.4061,0.9796,0.9796,NaN,10647.8345,0.3884,1.0589
5,spx,EGARCH,Student,0.1013,-0.1504,-10.0432,0.9847,0.9847,8.6487,10565.3809,0.4060,1.1546
6,tsx,GARCH,Normale,0.0751,NaN,NaN,0.9187,0.9938,NaN,10031.6664,0.3551,1.6678
7,tsx,GARCH,Student,0.0815,NaN,NaN,0.9125,0.9941,9.1733,9944.8192,0.3529,1.7221
8,tsx,GJR,Normale,0.0089,0.0989,5.3508,0.9267,0.9850,NaN,9966.0800,0.3388,1.6807
9,tsx,GJR,Student,0.0118,0.1040,5.7305,0.9199,0.9837,10.2080,9894.8842,0.3377,1.7421


Asymétrie S&P 500 contre TSX. Le coefficient d'asymétrie γ est significatif pour les deux indices (|t| entre 5,7 et 10,0), mais plus grand pour le S&P 500 (GJR : 0,164 contre 0,104 ; EGARCH : −0,150 contre −0,085). L'effet de levier est donc plus marqué aux États-Unis qu'au Canada dans notre échantillon. Nous n'avons pas testé formellement la différence entre les deux indices.